# WE (Telecom Egypt) Intelligent Customer-Support Chatbot — RAG Case Study

This notebook walks through the full Retrieval-Augmented Generation (RAG) pipeline built for this
case study, end to end: scraping the official Telecom Egypt website, chunking and embedding the
content, storing it in a persistent vector database, retrieving relevant context for a customer's
question, and generating a grounded answer with Gemini. It also covers letting a customer upload
their own document (PDF, DOCX, TXT, HTML, or an image) and query it the same way.

**Pipeline overview**

```
te.eg pages --> scrape + clean --> chunk --> embed --> ChromaDB (persistent)
                                                            |
customer question --> normalize query --> vector search ---+
                                                            |
                                          relevant chunks --> grounded answer
```

The production app (`app.py`) wraps this exact pipeline in a Gradio interface with a small
animated avatar and a file-upload box. This notebook exists to show and explain the underlying
logic on its own, independent of the UI.


## 1. Setup

Install dependencies and load the Gemini API key from a local `.env` file (never hard-code API keys in a notebook).

In [3]:
!pip install -q requests beautifulsoup4 sentence-transformers chromadb google-genai python-dotenv pypdf python-docx


In [ ]:
import os
import re
from collections import Counter
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from google import genai
import chromadb
from sentence_transformers import SentenceTransformer

GOOGLE_API_KEY = ""

gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

## 2. Scraping the official website (te.eg)

The knowledge base is grounded in Telecom Egypt's own site, per the case study's requirement that
"all responses must be grounded and bounded by the Telecom Egypt website."

Steps:
1. From a category page (e.g. entertainment), discover every sub-service page link.
2. Pull the meaningful text lines (headings, paragraphs, list items) from each page.
3. Any line that repeats across *more than one* page is nav/footer boilerplate (menus, copyright,
   "contact us", etc.) — drop it automatically instead of hand-listing every site's boilerplate.
4. Chunk the remaining clean content into ~500-character pieces, keeping sentences intact.


In [12]:
def scrape_category(category_url: str, category_name: str) -> list[dict]:
    """Full pipeline for one category page: discover -> extract -> clean -> chunk."""
    service_links = discover_service_links(category_url)
    all_pages_lines = {link: extract_raw_lines(link) for link in service_links}
    boilerplate = build_boilerplate_set(all_pages_lines.values())

    knowledge_base = []
    for url, lines in all_pages_lines.items():
        clean_lines = [l for l in lines if l not in boilerplate and not is_noise(l)]
        if clean_lines:
            knowledge_base.append({"url": url, "content": " ".join(clean_lines)})

    final_chunks = []
    for item in knowledge_base:
        page_chunks = chunk_page(item["content"])
        for i, chunk_text in enumerate(page_chunks):
            final_chunks.append({
                "chunk_id": f"{category_name}_{item['url'].split('/')[-1]}_{i}",
                "url": item["url"],
                "text": chunk_text,
                "category": category_name,
                "source_type": "website",
            })
    return final_chunks

In [14]:
CATEGORIES = {
    "entertainment": "https://te.eg/web/guest/personal/services/entertainment",
}

all_chunks = []
for category_name, category_url in CATEGORIES.items():
    print(f"Scraping category: {category_name}")
    chunks = scrape_category(category_url, category_name)
    print(f"  -> got {len(chunks)} chunks")
    all_chunks.extend(chunks)

Scraping category: entertainment
  -> got 12 chunks


## 3. Embedding + persistent vector storage

Each text chunk is embedded with a multilingual sentence-transformer model (works for both Arabic
and English) and stored in ChromaDB. We use `PersistentClient` (writes to disk) rather than the
default in-memory client, so the knowledge base survives an app restart — required for on-premises
deployment instead of a notebook-only demo.

In [15]:
CHROMA_DB_DIR = "./chroma_db"
COLLECTION_NAME = "we_knowledge_base"
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_DIR)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

def add_chunks(chunks: list[dict]):
    """Embed and upsert a list of chunk dicts into the collection."""
    if not chunks:
        return
    texts = [c["text"] for c in chunks]
    embeddings = embedding_model.encode(texts, show_progress_bar=True).tolist()
    collection.upsert(
        ids=[c["chunk_id"] for c in chunks],
        embeddings=embeddings,
        documents=texts,
        metadatas=[
            {
                "url": c.get("url", ""),
                "category": c.get("category", ""),
                "source_type": c.get("source_type", "website"),
            }
            for c in chunks
        ],
    )
    print(f"Upserted {len(chunks)} chunks. Collection total: {collection.count()}")

add_chunks(all_chunks)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Upserted 12 chunks. Collection total: 12


## 4. Letting customers upload their own documents

Per the case study, a customer should also be able to upload a document (PDF, DOCX, TXT, HTML, or
an image) and ask questions about *its* content, alongside the website knowledge.

Each format is handled differently:
- **PDF / DOCX / TXT / HTML** — text is extracted directly (via `pypdf`, `python-docx`,
  plain read, and `BeautifulSoup` respectively).
- **Images** — rather than a local OCR engine (which needs a separate system-level install,
  e.g. Tesseract, and doesn't ship the same way across OSes), the image is sent directly to
  Gemini's multimodal endpoint, which reads the text (or describes the image) natively.

Uploaded chunks are tagged `source_type="user_upload"` so retrieval can tell them apart from
scraped website content.

In [16]:
from pathlib import Path

def extract_pdf(path: str) -> str:
    from pypdf import PdfReader
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def extract_docx(path: str) -> str:
    import docx
    doc = docx.Document(path)
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    for table in doc.tables:
        for row in table.rows:
            cells_ = [c.text.strip() for c in row.cells if c.text.strip()]
            if cells_:
                paragraphs.append(" | ".join(cells_))
    return "\n".join(paragraphs)

def extract_txt(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def extract_html(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    texts = [tag.get_text(strip=True, separator=" ") for tag in soup.find_all(["h1", "h2", "h3", "h4", "p", "li", "td"])]
    return "\n".join(t for t in texts if len(t) > 5)

def extract_image(path: str) -> str:
    """Uses Gemini's native multimodal input instead of local OCR."""
    with open(path, "rb") as f:
        image_bytes = f.read()
    ext = Path(path).suffix.lower().lstrip(".")
    mime_type = "image/jpeg" if ext in ("jpg", "jpeg") else "image/png"
    response = gemini_client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=[{
            "role": "user",
            "parts": [
                {"inline_data": {"mime_type": mime_type, "data": image_bytes}},
                {"text": "\u0627\u0633\u062a\u062e\u0631\u062c \u0643\u0644 \u0627\u0644\u0646\u0635 \u0627\u0644\u0645\u0643\u062a\u0648\u0628 \u0641\u064a \u0627\u0644\u0635\u0648\u0631\u0629 \u062f\u064a \u062d\u0631\u0641\u064a\u064b\u0627."},
            ],
        }],
    )
    return response.text or ""

EXTRACTORS = {
    ".pdf": extract_pdf, ".docx": extract_docx, ".txt": extract_txt,
    ".html": extract_html, ".htm": extract_html,
    ".png": extract_image, ".jpg": extract_image, ".jpeg": extract_image,
}

def process_uploaded_file(file_path: str) -> list[dict]:
    ext = Path(file_path).suffix.lower()
    extractor = EXTRACTORS.get(ext)
    if extractor is None:
        raise ValueError(f"Unsupported file type: {ext}")
    text = extractor(file_path)
    filename = Path(file_path).name
    chunks = chunk_page(text)
    return [
        {"chunk_id": f"upload_{filename}_{i}", "url": filename, "text": c,
         "category": "user_upload", "source_type": "user_upload"}
        for i, c in enumerate(chunks)
    ]



## 5. Retrieval-Augmented Generation

This is the core of the assistant. For every customer message:

1. **Normalize the query** — ask Gemini to strip the message down to its core WE-service topic,
   fixing typos and dropping irrelevant specifics (like a singer's name in "\u0643\u0648\u0644 \u062a\u0648\u0646 \u0639\u0645\u0631\u0648 \u062f\u064a\u0627\u0628"), which
   meaningfully improves retrieval recall over embedding the raw message.
2. **Retrieve** the closest chunks from ChromaDB using cosine similarity.
3. **Filter by relevance** — ChromaDB always returns its top-k results even when nothing is
   actually close; a distance threshold drops chunks that aren't genuinely relevant, so casual
   small talk doesn't get contaminated with unrelated WE content.
4. **Generate** the final answer with Gemini, given the (filtered) context, with instructions to
   only answer from that context and say so plainly when it doesn't know — this is what keeps
   every response "grounded and bounded by the Telecom Egypt website".

In [19]:
RELEVANCE_THRESHOLD = 0.55  # cosine distance above this = not actually relevant

SYSTEM_PROMPT = """You are a friendly customer support assistant for Telecom Egypt (WE).

- If the customer sends casual small talk (greetings, thanks, goodbye, etc.), respond naturally and briefly.
- If the customer asks about a WE service or price, answer ONLY using the information in the context below. If the answer is not in the context, say clearly that you do not have this information and suggest contacting WE customer service on 155.
- Cite the source URL/filename for any factual claim you make, in parentheses at the end of the relevant sentence.
- Never invent prices, USSD codes, or details that are not in the context.
- Always answer in the same language as the customer's question (Arabic or English)."""

QUERY_NORMALIZE_PROMPT = """Extract the core WE (Telecom Egypt) service/topic being asked about, fixing any spelling mistakes, and dropping unrelated specifics (like a person's name) that a semantic search over WE service pages would not need.
Reply with ONLY the short core topic (1-4 words). If it is just small talk with no WE topic, reply with exactly: NONE

Message: {query}
Core topic:"""

def normalize_query(query: str) -> str:
    try:
        response = gemini_client.models.generate_content(
            model="gemini-flash-lite-latest",
            contents=QUERY_NORMALIZE_PROMPT.format(query=query),
        )
        cleaned = (response.text or "").strip()
        return query if (not cleaned or cleaned.upper() == "NONE") else cleaned
    except Exception:
        return query

def search(query: str, top_k: int = 5):
    query_embedding = embedding_model.encode([query]).tolist()
    return collection.query(query_embeddings=query_embedding, n_results=top_k)

def generate_answer(query: str, top_k: int = 5) -> str:
    search_query = normalize_query(query)
    results = search(search_query, top_k=top_k)

    documents = results["documents"][0] if results["documents"] else []
    distances = results["distances"][0] if results.get("distances") else []
    metadatas = results["metadatas"][0] if results.get("metadatas") else []

    relevant = [
        (doc, meta) for doc, dist, meta in zip(documents, distances, metadatas)
        if dist <= RELEVANCE_THRESHOLD
    ]

    if relevant:
        context = "\n\n".join(f"[{m.get('url', 'unknown')}] {d}" for d, m in relevant)
    else:
        context = "(no matching WE content found for this query)"

    prompt = f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nCustomer question: {query}"
    response = gemini_client.models.generate_content(model="gemini-flash-lite-latest", contents=prompt)
    return response.text


## 6. Trying it end to end

A few example queries covering casual small talk, a well-covered WE service question, and a question the knowledge base doesn't have an answer for.

In [20]:
test_queries = [
    "hello",                                  # small talk -> should reply naturally, no WE content needed
    "\u0639\u0627\u064a\u0632 \u0627\u0639\u0631\u0641 \u0633\u0639\u0631 \u0627\u0644\u0643\u0648\u0644 \u062a\u0648\u0646",             # covered by scraped entertainment content
    "\u0639\u0627\u064a\u0632 \u0627\u0639\u0631\u0641 \u0633\u0639\u0631 \u062e\u0637 \u0627\u0644\u0625\u0646\u062a\u0631\u0646\u062a \u0627\u0644\u0645\u0646\u0632\u0644\u064a",  # a category likely not scraped yet -> should say "don't know"
]

for q in test_queries:
    print("Q:", q)
    print("A:", generate_answer(q))
    print("-" * 60)


Q: hello
A: Hello! Welcome to Telecom Egypt (WE). How can I help you today?
------------------------------------------------------------
Q: عايز اعرف سعر الكول تون
A: أهلاً بحضرتك! 

عذراً، لا توجد لدي معلومات بخصوص سعر الكول تون في الوقت الحالي. أقترح عليك التواصل مع خدمة عملاء WE على الرقم 155 لمساعدتك في هذا الطلب. 

هل يمكنني مساعدتك في أي شيء آخر يخص خدماتنا المذكورة؟
------------------------------------------------------------
Q: عايز اعرف سعر خط الإنترنت المنزلي
A: أهلاً بك! عذراً، لا تتوفر لدي معلومات حول أسعار خطوط الإنترنت المنزلي في الوقت الحالي. يمكنك التواصل مع خدمة عملاء WE على الرقم 155 لمعرفة كافة التفاصيل. هل يمكنني مساعدتك بشيء آخر يخص خدماتنا؟
------------------------------------------------------------


## 7. From notebook to production

This notebook demonstrates the pipeline interactively. The deployable version of the same logic
lives in a set of standalone Python modules (no Colab dependency), run as a Gradio web app:

| Module | Responsibility |
|---|---|
| `scraper.py` | Section 2 above |
| `vector_store.py` | Section 3 above |
| `document_loader.py` | Section 4 above |
| `rag.py` | Section 5 above |
| `ingest.py` | One-off script that runs the scraper + vector store for every category |
| `app.py` | Gradio interface: chat box, file upload, animated avatar reflecting the bot's state (thinking / talking / confused) |

**On-premises deployment approach:**
- `chromadb.PersistentClient` writes the vector index to disk (`./chroma_db`), so the knowledge
  base survives restarts without needing an external managed vector DB.
- The Gemini API key is read from a `.env` file via `python-dotenv`, never hard-coded, and `.env`
  is excluded from version control via `.gitignore`.
- `app.py` runs as a standalone process (`python app.py`) exposing a Gradio server on a
  configurable host/port, suitable for running behind a reverse proxy on an on-prem VM or inside
  a Docker container — no cloud-specific (e.g. Colab) dependencies remain anywhere in the stack.
- Re-running `ingest.py` re-scrapes and upserts (not duplicates) into the same collection, so the
  knowledge base can be refreshed on a schedule without downtime.
